In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.nn import functional as F

from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

import numpy as np

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

Device: cuda


In [3]:
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

### Przygotowanie danych

In [ ]:
from utils import SinogramNoise

In [ ]:
from dival import get_standard_dataset

dataset = get_standard_dataset(
    "custom",
    data_path="../data/ct_reconstruction_dataset_128",
    sinogram_shape=(256, 183),
    image_shape=(128, 128),
    parts_len={"train": 203727, "validation": 26982, "test": 27236},
    impl="skimage",
)

transform_train = SinogramNoise(mean=0.0, std=0.0, p=1.0)
transform_test = SinogramNoise(mean=0.0, std=0.0, p=1.0)

train_dataset = dataset.create_torch_dataset(part="train", transform=transform_train)
test_dataset = dataset.create_torch_dataset(part="test", transform=transform_test)
val_dataset = dataset.create_torch_dataset(part="validation", transform=transform_test)

In [ ]:
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

#### Obliczenie odchylenia standardowego szumu przy augmentacji sinogramów

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def compute_dataset_signal_power(loader):
    total_power = 0.0
    total_count = 0

    for sinograms, _ in tqdm(loader, desc="Computing signal power", unit="batch"):
        # sinograms: [B, ...]
        total_power += sinograms.pow(2).sum().item()
        total_count += sinograms.numel()

    return total_power / total_count


def std_from_dataset(loader, target_snr_db):
    signal_power = compute_dataset_signal_power(loader)
    std = (signal_power / (10 ** (target_snr_db / 10))) ** 0.5
    return std

In [ ]:
TARGET_SNR_DB = 45.0

global_std = std_from_dataset(train_loader, TARGET_SNR_DB)
transform_train.std = global_std

print("Noise transform std:", global_std)

### Model Pix2Pix

In [4]:
from models.Pix2Pix_128_V2 import UnetGenerator, ConditionalDiscriminator

In [6]:
class GeneratorLoss(nn.Module):
    def __init__(self, alpha=100, beta=100):
        super().__init__()

        self.alpha = alpha
        self.beta = beta

        self.bce = nn.BCEWithLogitsLoss()
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()

    def forward(self, fake, real, fake_pred):
        fake_target = torch.ones_like(fake_pred)
        loss = (
            self.bce(fake_pred, fake_target)
            + self.alpha * self.l1(fake, real)
            + self.beta * self.mse(fake, real)
        )
        return loss


class DiscriminatorLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, fake_pred, real_pred):
        fake_target = torch.zeros_like(fake_pred)
        real_target = torch.ones_like(real_pred)
        fake_loss = self.loss_fn(fake_pred, fake_target)
        real_loss = self.loss_fn(real_pred, real_target)
        loss = (fake_loss + real_loss) / 2
        return loss

### Trening modelu

In [ ]:
generator = UnetGenerator().to(device)
discriminator = ConditionalDiscriminator().to(device)

g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.0001)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0001)

g_criterion = GeneratorLoss(alpha=100, beta=100)
d_criterion = DiscriminatorLoss()

Schedulery

In [8]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Number of trainable parameters generator:", count_parameters(generator))
print("Number of trainable parameters discriminator:", count_parameters(discriminator))

Number of trainable parameters generator: 54483713
Number of trainable parameters discriminator: 6829441


Precyzja połówkowa - scalery

In [11]:
from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

scaler_g = GradScaler(device="cuda")
scaler_d = GradScaler(device="cuda")

In [ ]:
import os


def save_state(epoch, generator, discriminator, g_optimizer, d_optimizer, scaler_g=None, scaler_d=None):
    checkpoint_dir = "checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint = {
        "epoch": epoch,
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "g_optimizer_state_dict": g_optimizer.state_dict(),
        "d_optimizer_state_dict": d_optimizer.state_dict(),
    }
    if scaler_g is not None:
        checkpoint["scaler_g_state_dict"] = scaler_g.state_dict()
    if scaler_d is not None:
        checkpoint["scaler_d_state_dict"] = scaler_d.state_dict()

    torch.save(checkpoint, f"{checkpoint_dir}/checkpoint_{epoch}.pt")

In [ ]:
def load_checkpoint(checkpoint_path, generator, discriminator, g_optimizer, d_optimizer, scaler_g=None, scaler_d=None, device='cuda'):
    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint at path: {checkpoint_path}")
        return 0

    print(f"Loading checkpoint: {checkpoint_path}...")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)

    generator.load_state_dict(checkpoint["generator_state_dict"])
    discriminator.load_state_dict(checkpoint["discriminator_state_dict"])

    g_optimizer.load_state_dict(checkpoint["g_optimizer_state_dict"])
    d_optimizer.load_state_dict(checkpoint["d_optimizer_state_dict"])

    if scaler_g is not None and "scaler_g_state_dict" in checkpoint:
        scaler_g.load_state_dict(checkpoint["scaler_g_state_dict"])
    if scaler_d is not None and "scaler_d_state_dict" in checkpoint:
        scaler_d.load_state_dict(checkpoint["scaler_d_state_dict"])

    epoch = checkpoint["epoch"]
    
    return epoch + 1

In [ ]:
from utils import calculate_mse, calculate_psnr, calculate_ssim, calculate_correlation

In [ ]:
num_epochs = 31
patience = num_epochs

best_loss = float("inf")
early_stopping_counter = 0

train_losses = []

# --- TRAIN METRICS ---
mse_train_losses = []
ssim_train_losses = []
psnr_train_losses = []
correlation_train_losses = []

# --- VAL METRICS ---
mse_val_losses = []
ssim_val_losses = []
psnr_val_losses = []
correlation_val_losses = []

# --- VAL LOSSES ---
g_val_losses = []
d_val_losses = []

# Create or clear log file
with open("training_log.txt", "w") as f:
    f.write("")

print("Training started")

for epoch in range(num_epochs):
    ge_loss = 0.0
    de_loss = 0.0

    mse_train = 0.0
    ssim_train = 0.0
    psnr_train = 0.0
    corr_train = 0.0

    generator.train()
    discriminator.train()

    # Training loop with progress bar
    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=False,
    )
    for sino, img in train_bar:
        sino = sino.unsqueeze(1).to(device, non_blocking=True)
        img = img.unsqueeze(1).to(device, non_blocking=True)

        # --- Generator ---
        fake = generator(sino)
        fake_pred = discriminator(fake, sino)
        g_loss = g_criterion(fake, img, fake_pred)

        g_optimizer.zero_grad(set_to_none=True)
        g_loss.backward()
        g_optimizer.step()

        # --- Discriminator ---
        fake_detached = fake.detach()
        fake_pred = discriminator(fake_detached, sino)
        real_pred = discriminator(img, sino)
        d_loss = d_criterion(fake_pred, real_pred)

        d_optimizer.zero_grad(set_to_none=True)
        d_loss.backward()
        d_optimizer.step()

        ge_loss += g_loss.item()
        de_loss += d_loss.item()

        # ----- TRAIN METRICS -----
        mse_train += calculate_mse(fake, img)
        ssim_train += calculate_ssim(fake, img)
        psnr_train += calculate_psnr(fake, img)

        corr_train += np.corrcoef(
            img.squeeze().detach().cpu().numpy().flatten(),
            fake.squeeze().detach().cpu().numpy().flatten(),
        )[0, 1]

    # --- Average TRAIN ---
    ge_loss /= len(train_loader)
    de_loss /= len(train_loader)

    mse_train /= len(train_loader)
    ssim_train /= len(train_loader)
    psnr_train /= len(train_loader)
    corr_train /= len(train_loader)

    # ================= VALIDATION =================
    generator.eval()
    discriminator.eval()

    mse_val = 0.0
    ssim_val = 0.0
    psnr_val = 0.0
    corr_val = 0.0

    g_val_loss = 0.0
    d_val_loss = 0.0

    with torch.inference_mode():
        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{num_epochs} [Val]",
            leave=False,
        )

        for sino, img in val_bar:

            sino = sino.unsqueeze(1).to(device, non_blocking=True)
            img = img.unsqueeze(1).to(device, non_blocking=True)

            output = generator(sino)

            # --- Generator val loss ---
            fake_pred = discriminator(output, sino)
            g_val_loss += g_criterion(output, img, fake_pred).item()

            # --- Discriminator val loss ---
            fake_pred = discriminator(output.detach(), sino)
            real_pred = discriminator(img, sino)
            d_val_loss += d_criterion(fake_pred, real_pred).item()

            # --- METRICS ---
            mse_val += calculate_mse(output, img)
            ssim_val += calculate_ssim(output, img)
            psnr_val += calculate_psnr(output, img)

            corr_val += np.corrcoef(
                img.squeeze().cpu().numpy().flatten(),
                output.squeeze().cpu().numpy().flatten(),
            )[0, 1]

    # --- Average VAL ---
    g_val_loss /= len(val_loader)
    d_val_loss /= len(val_loader)

    mse_val /= len(val_loader)
    ssim_val /= len(val_loader)
    psnr_val /= len(val_loader)
    corr_val /= len(val_loader)

    # ================= SAVE =================
    if epoch % 5 == 0:
        save_state(
            epoch, generator, discriminator, g_optimizer, d_optimizer
        )

    # Best model by VAL MSE
    if mse_val < best_loss:
        best_loss = mse_val
        torch.save(generator.state_dict(), "best_generator.pth")

    # ================= EARLY STOP =================
    if mse_val > (
        1.01 * mse_val_losses[-1] if mse_val_losses else float("inf")
    ):
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print("Early stopping triggered")
        break

    # ================= STORE HISTORY =================
    train_losses.append(ge_loss)

    mse_train_losses.append(mse_train)
    ssim_train_losses.append(ssim_train)
    psnr_train_losses.append(psnr_train)
    correlation_train_losses.append(corr_train)

    mse_val_losses.append(mse_val)
    ssim_val_losses.append(ssim_val)
    psnr_val_losses.append(psnr_val)
    correlation_val_losses.append(corr_val)

    g_val_losses.append(g_val_loss)
    d_val_losses.append(d_val_loss)

    # ================= LOG =================
    log = (
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {(ge_loss + de_loss):.4f} | "
        f"Val Loss: {(g_val_loss + d_val_loss):.4f} | "
        f"Train MSE: {mse_train:.6f} | "
        f"Val MSE: {mse_val:.6f} | "
        f"Train SSIM: {ssim_train:.4f} | "
        f"Val SSIM: {ssim_val:.4f} | "
        f"Train PSNR: {psnr_train:.4f} | "
        f"Val PSNR: {psnr_val:.4f} | "
        f"Train Corr: {corr_train:.4f} | "
        f"Val Corr: {corr_val:.4f}"
    )

    print(log)
    with open("training_log.txt", "a") as f:
        f.write(log + "\n")

print("Training completed")

### Testowanie, zapisanie wag

In [ ]:
import matplotlib.pyplot as plt
import os

epochs = range(1, len(train_losses) + 1)

# ================= LOSS =================
plt.figure()
plt.plot(epochs, train_losses, label="Train Total Loss")
plt.plot(epochs, g_val_losses, label="G Val Loss")
plt.plot(epochs, d_val_losses, label="D Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()
plt.show()


# ================= MSE =================
plt.figure()
plt.plot(epochs, mse_train_losses, label="Train MSE")
plt.plot(epochs, mse_val_losses, label="Val MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("MSE")
plt.legend()
plt.grid()
plt.show()


# ================= SSIM =================
plt.figure()
plt.plot(epochs, ssim_train_losses, label="Train SSIM")
plt.plot(epochs, ssim_val_losses, label="Val SSIM")
plt.xlabel("Epoch")
plt.ylabel("SSIM")
plt.title("SSIM")
plt.legend()
plt.grid()
plt.show()


# ================= PSNR =================
plt.figure()
plt.plot(epochs, psnr_train_losses, label="Train PSNR")
plt.plot(epochs, psnr_val_losses, label="Val PSNR")
plt.xlabel("Epoch")
plt.ylabel("PSNR")
plt.title("PSNR")
plt.legend()
plt.grid()
plt.show()


# ================= CORRELATION =================
plt.figure()
plt.plot(epochs, correlation_train_losses, label="Train Corr")
plt.plot(epochs, correlation_val_losses, label="Val Corr")
plt.xlabel("Epoch")
plt.ylabel("Correlation")
plt.title("Correlation")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def load_generator(checkpoint_path, generator, device="cuda"):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(
            f"No checkpoint found at {checkpoint_path}"
        )

    checkpoint = torch.load(checkpoint_path, map_location=device)

    generator.load_state_dict(checkpoint["generator_state_dict"])

    epoch = checkpoint.get("epoch", None)
    print(
        f"Loaded generator from '{checkpoint_path}' (epoch {epoch})"
    )


load_generator("checkpoints/checkpoint_10.pt", generator, device)

In [ ]:
# Test loop with progress bar
generator.eval()
mse_test_loss = 0.0
ssim_test_loss = 0.0
psnr_test_loss = 0.0
with torch.inference_mode():
    with autocast(device_type="cuda"):
        val_bar = tqdm(test_loader, desc=f"[Test]", leave=False)
        for sino, img in val_bar:
            sino = sino.unsqueeze(1).to(device, non_blocking=True)
            img = img.unsqueeze(1).to(device, non_blocking=True)

            # MSE val loss
            output = generator(sino)
            mse_test_loss += calculate_mse(output, img)

            # SSIM val loss
            ssim_test_loss += calculate_ssim(output, img)

            # PSNR val loss
            psnr_test_loss += calculate_psnr(output, img)

mse_test_loss /= len(test_loader)
ssim_test_loss /= len(test_loader)
psnr_test_loss /= len(test_loader)

print(
    f"Test MSE: {mse_test_loss:.6f} | "
    f"Test SSIM: {ssim_test_loss:.4f} | "
    f"Test PSNR: {psnr_test_loss:.4f}"
)